In [1]:
using Revise
using InteractiveUtils

const PATH_METADATA_ENRICH_JL = "functions/enrich_metadata.jl"
# const PATH_PDF_EXTRACT_JL = "functions/qog_pdf_extract.jl"
# const PATH_METADATA_ENHANCE_JL = "functions/qog_metadata_join.jl"

# Print a summary of all dataframes that are current loaded in Main
function dataframe_summaries(mod=Main)
    for n in names(mod)
        x = getfield(mod, n)
        if x isa AbstractDataFrame
            println("=== DataFrame: ", n, " ===")
            println(summary(x))
            println()
        end
    end
end


includet(PATH_METADATA_ENRICH_JL)

QoG Time-Series Loader Pipeline
Input: data/qog_std_ts_jan25.arrow

>>> Step 1/5: Loading raw data with identity promotion...
    Loaded: 12391 rows × 2010 columns
    ✓ ggis_rowid assigned

>>> Step 2/5: Previewing rescue collisions...
>>> RESCUE COLLISION PREVIEW (informational — NO rows will be deleted):
    Total (ccode, year) pairs with >1 row: 22

    By entity combination:
      VDR + VNM: 22 years (1955-1976)
        Years: 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976

    NOTE: Use `ggis_rowid` as unique key, or (ident_ccode, ident_year, ident_ccodealp)
    ⚠️  Found 1 collision group(s)
    (See year details above)

>>> Step 3/5: Rescuing historical ccodes...
>>> Historical Ccode Rescue:
    Missing before: 234
    Rescued: 234
    Missing after: 0
    Row count: 12391 (unchanged)
    By alpha code:
      ETH → 231: 47 rows
      YEM → 887: 44 rows
      DEU → 276: 42 rows
      MHL → 584: 3

In [15]:

dataframe_summaries()

=== DataFrame: REGION_LABELS ===
10×2 DataFrame

=== DataFrame: df ===
12391×2013 DataFrame

=== DataFrame: meta_df ===
2010×8 DataFrame



In [19]:
meta_plus1 = enrich_metadata_with_lifespan();


>>> Results Summary
    Variables Audited: 2009
      :modern — 887
      :experimental — 355
      :legacy — 240
      :current — 240
      :historical — 168
      :anchor — 119


In [20]:
check_year_discrepancies(meta_plus1)

No year discrepancies found exceeding tolerance 1.


In [22]:
meta_plus2 = enrich_metadata_with_geographic_coverage(meta_plus1);


>>> Computing Geographic Coverage (Step 8)
    Global Threshold:   ≥ 0.95
    Regional Threshold: ≥ 0.95 (Core) AND ≤ 0.05 (Exclusion in 7+ regions)
    Pre-computing geographic universes...

>>> Geographic Classification Summary
    :other — 1310
    :global — 616
    :regional — 83


In [23]:
CSV.write("data/qog_metadata_plus2.csv", meta_plus2)

"data/qog_metadata_plus2.csv"

In [39]:
is_global(x) =
    !ismissing(x) &&
    lowercase(strip(string(x))) == "global"

starts_wdi_gdp(x) =
    !ismissing(x) &&
    startswith(string(x), "wdi_gdp")

df2 = meta_plus2 |>
    x -> subset(
        x,
        :slug => ByRow(starts_wdi_gdp),
        :ggis_geo_classification => ByRow(is_global)
    ) |>
    x -> select(
        x,
        Not([:prefix, :description, :type, :provenance, :min_year, :max_year])
    )


Row,slug,label,ggis_birth_year,ggis_death_year,ggis_is_active,ggis_temporal_gap,ggis_temporal_profile,ggis_global_penetration,ggis_geo_classification,ggis_region_penetration
,String31,String?,Int64?,Int64?,Bool?,Bool?,Symbol?,Float64?,Symbol?,Array…?
1,wdi_gdpcapcon2015,GDP per capita (constant 2015 US dollar),1960,2023,true,false,modern,0.975799,global,"[1.0, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
2,wdi_gdpcapcur,GDP per capita (current US dollar),1960,2023,true,false,modern,0.975799,global,"[1.0, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
3,wdi_gdpcapgr,GDP per capita growth (annual %),1961,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
4,wdi_gdpcappppcon2021,"GDP per capita, PPP (constant 2021 international dollar)",1990,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
5,wdi_gdpcappppcur,"GDP per capita, PPP (current international dollar)",1990,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
6,wdi_gdpgr,GDP growth (annual %),1961,2023,true,false,modern,0.975799,global,"[1.0, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
7,wdi_gdppppcon2021,"GDP, PPP (constant 2021 international dollar)",1990,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
8,wdi_gdppppcur,"GDP, PPP (current international dollar)",1990,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"


In [33]:
select(
    subset(
        meta_plus2,
        :slug => ByRow(s -> startswith(s, "wdi_gdp")),   # allow missing-handling by skipmissing
        :ggis_geo_classification => ByRow(==("global"));
        skipmissing=true
    ),
    Not([:label, :prefix, :description, :type, :provenance, :min_year, :max_year])
)

Row,slug,ggis_birth_year,ggis_death_year,ggis_is_active,ggis_temporal_gap,ggis_temporal_profile,ggis_global_penetration,ggis_geo_classification,ggis_region_penetration
,String31,Int64?,Int64?,Bool?,Bool?,Symbol?,Float64?,Symbol?,Array…?


In [32]:
select(
    subset(
        meta_plus2,
        :slug => ByRow(s -> !ismissing(s) && startswith(s, "wdi_gdp")),
        :ggis_geo_classification => ByRow(g -> !ismissing(g) && g == "global"),
    ),
    Not([:label, :prefix, :description, :type, :provenance, :min_year, :max_year])
)

Row,slug,ggis_birth_year,ggis_death_year,ggis_is_active,ggis_temporal_gap,ggis_temporal_profile,ggis_global_penetration,ggis_geo_classification,ggis_region_penetration
,String31,Int64?,Int64?,Bool?,Bool?,Symbol?,Float64?,Symbol?,Array…?


In [29]:
select(
    filter(:slug => s -> startswith(s, "wdi_gdp"), meta_plus2),
    Not(:label, :prefix, :description, :type, :provenance, :min_year, :max_year)
)

Row,slug,ggis_birth_year,ggis_death_year,ggis_is_active,ggis_temporal_gap,ggis_temporal_profile,ggis_global_penetration,ggis_geo_classification,ggis_region_penetration
,String31,Int64?,Int64?,Bool?,Bool?,Symbol?,Float64?,Symbol?,Array…?
1,wdi_gdpagr,1960,2023,true,false,modern,0.892904,other,"[0.928571, 0.85, 0.6, 0.877551, 0.777778, 0.5, 0.909091, 0.625, 0.25, 0.769231]"
2,wdi_gdpcapcon2015,1960,2023,true,false,modern,0.975799,global,"[1.0, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
3,wdi_gdpcapcur,1960,2023,true,false,modern,0.975799,global,"[1.0, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
4,wdi_gdpcapgr,1961,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
5,wdi_gdpcappppcon2021,1990,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
6,wdi_gdpcappppcur,1990,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
7,wdi_gdpgr,1961,2023,true,false,modern,0.975799,global,"[1.0, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"
8,wdi_gdpind,1960,2023,true,false,modern,0.892904,other,"[0.928571, 0.85, 0.6, 0.877551, 0.777778, 0.5, 0.909091, 0.625, 0.25, 0.769231]"
9,wdi_gdppppcon2021,1990,2023,true,false,modern,0.974882,global,"[0.964286, 0.9, 0.8, 0.938776, 0.888889, 0.666667, 1.0, 0.75, 0.916667, 1.0]"


In [18]:
run_enrich_metadata_samples();


  enrich_metadata.jl — Function intent and usage

┌─ load_dataframes()
│  INTENT: Load the main QoG timeseries and the joined metadata in one call.
│  USE WHEN: You need both df and meta_df for auditing or enrichment.
│
│  RETURNS: (df, meta_df)
│    - df: main timeseries from load_qog_timeseries()
│    - meta_df: from PATH_METADATA_JOINED
│
│  USAGE:
│    df, meta = load_dataframes()
└──────────────────────────────────────────────────────────────────────────

┌─ classify_temporal_profile(birth_year, death_year; kwargs...)
│  INTENT: Classify a variable's temporal profile from its lifespan.
│  USE WHEN: You have first/last year and want :anchor, :experimental,
│           :legacy, :historical, :current, :modern, or :unclassified.
│
│  ARGUMENTS:
│    birth_year::Int, death_year::Int (positional)
│    Optional kwargs: data_start, data_end, current_year, active_lag, thresholds
│
│  RETURNS: Symbol (e.g. :anchor, :current)
│
│  USAGE:
│    profile = classify_temporal_profile(1946, 2022)
